# Phase 7 - Stage C attempt 4: RESUME `stage_c_run4` (epoch 41 -> 50)

Resumes the 50-epoch Stage C fine-tune from `phase7/stage_c_run4/latest.pt`
(epoch 40 done, 1050 batches into epoch 41, global step 81050), then runs the
planned post-run coherence decay and the Task 7.2c benchmark.

**Why a separate notebook:** the original run-4 launch cell only lived in the
Drive copy of `phase7_stage_c_colab.ipynb`; the repo's cell 7 is the stale
attempt-3b cell (writes to `stage_c_run3`). Running it would NOT resume run 4.

**Safety model** (see `experiments/train_universal.py:1038-1067`): if a resume
fails for any reason the trainer silently starts a FRESH run that wipes
`metrics.jsonl` / `metrics_by_task.csv` / `steps.jsonl` and later overwrites
`latest.pt` / `best.pt`. This notebook therefore
1. backs up `stage_c_run4` first (cell 2),
2. generates every training flag from the checkpoint's own saved
   `training_config` / `model_config` and checks them with the trainer's own
   parser and config functions before launching (cell 6),
3. kills the trainer the moment it prints `Could not resume` (cell 7).

**Data:** the original manifests / Stage B / coherence checkpoints were not
handed over, so datasets are re-downloaded from public sources and manifests
rebuilt (seed 2027, 10% validation). The resume uses
`--resume-ignore-config-mismatch`; only the manifest / init / dictionary
path+hash keys are allowed to differ.

**Images are kept on the Colab VM's local disk (`/content/datasets`)**, not on
Drive: downloads take minutes instead of hours, and training no longer dies
when the Drive mount drops. Checkpoints, logs and the manifests stay on Drive.

**Every new Colab session: run cells 1, 3, 4, 4b, 5, 6, 7 in order** (cell 4
re-downloads the images to the fresh VM disk; cell 5 reuses the manifests
already on Drive, so the training data stays identical across sessions).
Cell 2 (backup) only needs to run once.

In [ ]:
# @title 1) Mount Drive, clone repo, paths
from pathlib import Path
import os
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
REPO = Path('/content/SparseDict')
DRIVE_ROOT = Path('/content/drive/MyDrive/SparseDict')
# Images live on the Colab VM's local disk: much faster to download and immune to
# Drive mount drops during training. The VM disk is wiped when the session ends,
# so cells 3-5 re-download each new session (minutes). Checkpoints, logs and the
# manifests stay on Drive.
DATA_ROOT = Path('/content/datasets')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = DRIVE_ROOT / 'phase7' / 'stage_c_coherence_retrain'   # rebuilt manifests live here
RUN4 = DRIVE_ROOT / 'phase7' / 'stage_c_run4'                     # checkpoints being resumed
TRAIN_MANIFEST = RUN_DIR / 'train.csv'
VALIDATION_MANIFEST = RUN_DIR / 'validation.csv'
FAILURE_LOG = RUN4 / 'stage_c_failure.log'
DEVICE = 'cuda'
RUN_DIR.mkdir(parents=True, exist_ok=True)
if not (RUN4 / 'latest.pt').is_file():
    raise FileNotFoundError(f'{RUN4}/latest.pt not found - upload the stage_c_run4 folder to MyDrive/SparseDict/phase7/ first.')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'master', 'https://github.com/kalpeshjp89/SparseDict.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'master'], check=True)
print('repo HEAD:', subprocess.run(['git', '-C', str(REPO), 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())
sys.path.insert(0, str(REPO))
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab before training.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# @title 2) Back up stage_c_run4 before anything touches it
import shutil
import time
existing = sorted(RUN4.parent.glob(RUN4.name + '_backup_*'))
if existing:
    print('Backup already exists, not creating another:', existing[-1])
else:
    dest = RUN4.parent / (RUN4.name + '_backup_' + time.strftime('%Y%m%d_%H%M%S'))
    shutil.copytree(RUN4, dest)
    print('Backed up', RUN4, '->', dest)
    print(sorted(p.name for p in dest.iterdir()))
# To restore after an accidental fresh run:
#   shutil.rmtree(RUN4); shutil.copytree(<backup dir>, RUN4)

In [ ]:
# @title 3) Kaggle credentials (needed for the SOTS download)
if (DATA_ROOT / 'SOTS').is_dir():
    print('SOTS already on Drive; skipping Kaggle credential setup.')
else:
    # Kaggle token setup - paste ONCE per runtime. On kaggle.com go to
    # Account -> API -> "Create New Token"; the downloaded kaggle.json contains
    # exactly two values: username and key. Input below is masked with getpass
    # and is never written into the notebook file.
    import json
    import os
    import subprocess
    import sys
    from pathlib import Path


    def _colab_secret(name: str) -> str:
        try:
            from google.colab import userdata

            return userdata.get(name)
        except Exception:
            return ""


    KAGGLE_USERNAME_INPUT = _colab_secret("KAGGLE_USERNAME")
    KAGGLE_API_KEY_INPUT = _colab_secret("KAGGLE_API_KEY")
    if not (KAGGLE_USERNAME_INPUT and KAGGLE_API_KEY_INPUT):
        import getpass

        KAGGLE_USERNAME_INPUT = getpass.getpass("Kaggle username: ").strip()
        KAGGLE_API_KEY_INPUT = getpass.getpass("Kaggle API key: ").strip()

    KAGGLE_READY = False
    if not (KAGGLE_USERNAME_INPUT and KAGGLE_API_KEY_INPUT):
        print(
            "Kaggle credentials skipped. The `sots` kaggle: source below will "
            "fail until this cell is completed with username + key."
        )
    else:
        kaggle_dir = Path.home() / ".kaggle"
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        token_path = kaggle_dir / "kaggle.json"
        token_path.write_text(
            json.dumps({"username": KAGGLE_USERNAME_INPUT, "key": KAGGLE_API_KEY_INPUT})
        )
        token_path.chmod(0o600)
        os.environ["KAGGLE_CONFIG_DIR"] = str(kaggle_dir)
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "kaggle"]
        ).check_returncode()
        result = subprocess.run(
            [
                "kaggle",
                "datasets",
                "files",
                "balraj98/synthetic-objective-testing-set-sots-reside",
            ],
            capture_output=True,
            text=True,
        )
        KAGGLE_READY = result.returncode == 0
        print(
            "kaggle auth check:",
            "OK" if KAGGLE_READY else (result.stderr or result.stdout).strip()[:300],
        )
        if KAGGLE_READY:
            print(
                "Kaggle credentials installed at ~/.kaggle/kaggle.json. If the "
                "check above reports a 403/permission error instead, accept the "
                "dataset once: open the dataset page on Kaggle while signed in "
                "and agree to its access rule, then rerun this cell."
            )

In [ ]:
# @title 4) Download the datasets to local disk (/content/datasets), one corpus per session if space is tight
# Small -> large order. Already-downloaded folders on Drive are reused, so a
# rerun only fetches what is missing. Keep 'flickr2k' (10.85 GB HR zip) out
# of the list unless this session has room for it.
DATASETS = ['cbsd68', 'sots', 'outdoor-rain', 'div2k']
DOWNLOAD_PUBLIC_DATASETS = True
DOWNLOAD_URBAN100_FOR_BENCHMARK = True
# DIV2K is academic-research-only; explicitly accept its terms to download.
ACCEPT_DIV2K_TERMS = True  # academic research use confirmed
SOURCE_URLS = {
    # SOTS dehazing mirror (requires the Kaggle credentials cell to pass).
    'sots': 'kaggle:balraj98/synthetic-objective-testing-set-sots-reside',
    # Outdoor-Rain / deraining training pairs (Restormer's Rain13K protocol,
    # spread HF repository downloaded with huggingface_hub).
    'outdoor-rain': 'hf-dataset:dronefreak/Rain13K',
}
DRIVE_ARCHIVES = {}  # e.g. {'sots': '/content/drive/MyDrive/datasets/SOTS.zip'}
DRIVE_FILE_IDS = {}  # e.g. {'cbsd68': 'Google-Drive-file-id'}
if DOWNLOAD_URBAN100_FOR_BENCHMARK:
    DATASETS.append('urban100')  # 129 MB HR tar.gz from Hugging Face
command = [sys.executable, str(REPO / 'scripts' / 'prepare_phase7_datasets.py'),
           '--root', str(DATA_ROOT)]
for dataset in DATASETS:
    command.extend(['--dataset', dataset])
if DOWNLOAD_PUBLIC_DATASETS:
    command.append('--download-public')
if ACCEPT_DIV2K_TERMS:
    command.append('--accept-div2k-license')
for dataset, path in DRIVE_ARCHIVES.items():
    command.extend(['--drive-archive', f'{dataset}={path}'])
for dataset, file_id in DRIVE_FILE_IDS.items():
    command.extend(['--drive-file-id', f'{dataset}={file_id}'])
for dataset, url in SOURCE_URLS.items():
    command.extend(['--source-url', f'{dataset}={url}'])
import os
proc = subprocess.run(
    command, cwd=REPO, capture_output=True, text=True,
    env=dict(os.environ, PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True'),
)
print('returncode:', proc.returncode)
if proc.stdout:
    print('---- child stdout (tail) ----')
    print(proc.stdout[-6000:])
if proc.stderr:
    print('---- child stderr (tail) ----')
    print(proc.stderr[-6000:])
if proc.returncode != 0:
    raise RuntimeError('dataset download failed - rerun this cell (finished datasets are skipped).')
import shutil
_free_gb = shutil.disk_usage('/content').free / 1e9
print(f'Local disk free after downloads: {_free_gb:.1f} GB')

In [ ]:
# @title 4b) Make sure Rain13K's TRAIN targets are on disk (+ drop stale manifests)
from huggingface_hub import snapshot_download
RAIN_TRAIN = DATA_ROOT / 'Outdoor-Rain' / 'train' / 'target'
snapshot_download(repo_id='dronefreak/Rain13K', repo_type='dataset',
                  local_dir=str(DATA_ROOT / 'Outdoor-Rain'), allow_patterns=['train/target/*'])
n_rain = sum(1 for p in RAIN_TRAIN.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
print('Rain13K train targets on disk:', n_rain, '(expected 13711)')
assert n_rain > 13000, 'Rain13K train split incomplete - rerun this cell'
# Manifests built without the Rain13K train split are wrong: remove them so cell 5 rebuilds.
for manifest in (TRAIN_MANIFEST, VALIDATION_MANIFEST):
    if manifest.is_file() and 'Outdoor-Rain/train/target' not in manifest.read_text(encoding='utf8'):
        manifest.unlink()
        print('removed stale manifest (no Rain13K train images):', manifest)

In [ ]:
# @title 5) Build (or reuse) the train/validation manifests
from scripts.prepare_phase7_datasets import resolve_clean_image_dir
# Rebuilding after the first resume would change the training data mid-run;
# keep False once the CSVs exist unless you deliberately want to regenerate.
REBUILD_MANIFESTS = not (TRAIN_MANIFEST.is_file() and VALIDATION_MANIFEST.is_file())
TRAIN_DATASETS = [name for name in DATASETS if name != 'urban100']
# Pin Rain13K to its TRAIN split (the auto-resolver falls back to a 100-image test set if train is missing).
DATASET_PATH_OVERRIDES = {'outdoor-rain': DATA_ROOT / 'Outdoor-Rain' / 'train' / 'target'}
DATASET_DIRS = {
    name: Path(DATASET_PATH_OVERRIDES[name]) if name in DATASET_PATH_OVERRIDES
    else resolve_clean_image_dir(name, DATA_ROOT)
    for name in TRAIN_DATASETS
}
IMAGE_DIRS = list(DATASET_DIRS.values())
# Same SOTS handling as phase7_stage_c_colab.ipynb: add the other GT/clear halves.
_SOTS_ROOT = DATA_ROOT / 'SOTS'
_IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg', '.bmp'}
if _SOTS_ROOT.is_dir():
    for sub in sorted(
        p for p in _SOTS_ROOT.rglob('*')
        if p.is_dir() and p.name.casefold() in {'gt', 'clear', 'target', 'ground_truth'}
        and any(f.suffix.lower() in _IMAGE_SUFFIXES for f in p.rglob('*'))
    ):
        if sub not in IMAGE_DIRS:
            IMAGE_DIRS.append(sub)
if not all(path.is_dir() for path in IMAGE_DIRS):
    raise FileNotFoundError('Some dataset folders are missing; rerun cell 4.')

_all_images = set()
for directory in IMAGE_DIRS:
    found = {p.resolve() for p in directory.rglob('*') if p.suffix.lower() in _IMAGE_SUFFIXES}
    _all_images |= found
    print(f'{len(found):6d} images  {directory}')
print(f'{len(_all_images):6d} unique images total (original run: 13419)')

TASKS = 'sr2,sr3,sr4,denoise15,denoise25,denoise50,derain,desnow,dehaze,lowlight,jpeg'
if REBUILD_MANIFESTS:
    manifest_command = [
        sys.executable, str(REPO / 'scripts' / 'make_universal_manifest.py'),
        '--train-output', str(TRAIN_MANIFEST), '--validation-output', str(VALIDATION_MANIFEST),
        '--validation-fraction', '0.1', '--seed', '2027', '--tasks', TASKS,
    ]
    for directory in IMAGE_DIRS:
        manifest_command.extend(['--image-dir', str(directory)])
    subprocess.run(manifest_command, cwd=REPO, check=True)
else:
    print('Reusing existing manifests (set REBUILD_MANIFESTS = True to regenerate).')

import csv
def _manifest_paths(path):
    with open(path, encoding='utf8', newline='') as f:
        return [row['image_path'] for row in csv.DictReader(f)]
train_paths, val_paths = _manifest_paths(TRAIN_MANIFEST), _manifest_paths(VALIDATION_MANIFEST)
print(f'train rows {len(train_paths)} (original 132847) | validation rows {len(val_paths)} (original 14762)')
if (len(train_paths), len(val_paths)) != (132847, 14762):
    print('NOTE: the rebuilt image set differs from the original run (expected: the public '
          'Rain13K train split alone has 13,711 images, more than the original 13,419 total). '
          'The remaining epochs train on this rebuilt set - mention it to your senior.')
missing = sorted({p for p in train_paths + val_paths if not Path(p).is_file()})
if missing:
    raise FileNotFoundError(f'{len(missing)} manifest images are missing on this VM, e.g. {missing[:3]}. '
                            'Rerun cell 4 so every dataset is downloaded before training.')
print('All manifest images present on local disk.')

In [ ]:
# @title 6) Preflight: generate flags from latest.pt and verify them with the trainer's own code
import argparse
import importlib.util
import json

ckpt = torch.load(RUN4 / 'latest.pt', map_location='cpu', weights_only=False)
saved_tc, saved_mc = ckpt['training_config'], ckpt['model_config']
print(f"latest.pt: phase={ckpt['phase']} epoch={ckpt['epoch']} next_batch={ckpt['next_batch']} "
      f"step={ckpt['global_step']} best_psnr={ckpt['best_validation_psnr']:.3f}")
assert ckpt['phase'] == 'fine_tune', 'latest.pt is not a fine_tune checkpoint'
assert ckpt['global_step'] >= 81050 and ckpt['epoch'] >= 40, (
    'latest.pt is OLDER than the handed-over checkpoint (step 81050) - a fresh run may have '
    'overwritten it. Restore from the cell-2 backup before continuing.')
EXPECTED_RESUME = (f"Resuming fine_tune at epoch={ckpt['epoch'] + 1}, "
                   f"batch={ckpt['next_batch']}, step={ckpt['global_step']}")

# Keys that are allowed to differ: rebuilt manifests + Stage B / coherence files we do not have.
ALLOWED_DIFF = {'train_manifest_sha256', 'validation_manifest_sha256', 'init_checkpoint',
                'init_checkpoint_sha256', 'dictionary_checkpoint', 'dictionary_checkpoint_sha256'}
SKIP_KEYS = ALLOWED_DIFF | {'phase', 'train_manifest', 'validation_manifest', 'model_config'}
BOOL_KEYS = {'coherence_squeeze', 'normalize_task_losses'}

def build_args(tc, mc, train_manifest, validation_manifest, output_dir, device):
    argv = ['--phase', tc['phase'], '--train-manifest', str(train_manifest),
            '--validation-manifest', str(validation_manifest), '--output-dir', str(output_dir)]
    for key, value in tc.items():
        if key in SKIP_KEYS:
            continue
        flag = '--' + key.replace('_', '-')
        if key in BOOL_KEYS:
            argv.append(flag if value else '--no-' + key.replace('_', '-'))
        else:
            argv += [flag, str(value)]
    argv += ['--dim', str(mc['d']), '--atoms-per-expert', str(mc['atoms_per_expert']),
             '--top-k', str(mc['top_k']), '--aux-weight', str(mc['aux_weight'])]
    if mc['noise_std'] > 0:
        argv += ['--logit-jitter', str(mc['noise_std'])]
    argv += ['--epochs', '50', '--device', device, '--resume-ignore-config-mismatch']
    return argv

def load_trainer(repo):
    spec = importlib.util.spec_from_file_location('train_universal_preflight', repo / 'experiments' / 'train_universal.py')
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

def parse_with_trainer(tu, argv):
    # Run the trainer's real argparse definition but stop right after parsing.
    class _Parsed(Exception):
        pass
    original = argparse.ArgumentParser.parse_args
    def capture(self, *a, **k):
        raise _Parsed(original(self, *a, **k))
    argparse.ArgumentParser.parse_args = capture
    try:
        tu.main(argv)
    except _Parsed as parsed:
        return parsed.args[0]
    finally:
        argparse.ArgumentParser.parse_args = original
    raise RuntimeError('trainer main() returned without parsing arguments')

def config_diff(tu, args, tc, mc):
    new_tc, new_mc = tu._training_config(args), tu._model_config(args)
    keys = set(tc) | set(new_tc)
    tc_diff = {k: (tc.get(k), new_tc.get(k)) for k in sorted(keys) if tc.get(k) != new_tc.get(k)}
    mc_diff = {k: (mc.get(k), new_mc.get(k)) for k in sorted(set(mc) | set(new_mc)) if mc.get(k) != new_mc.get(k)}
    return tc_diff, mc_diff

ARGV = build_args(saved_tc, saved_mc, TRAIN_MANIFEST, VALIDATION_MANIFEST, RUN4, DEVICE)
tu = load_trainer(REPO)
parsed_args = parse_with_trainer(tu, ARGV)
tc_diff, mc_diff = config_diff(tu, parsed_args, saved_tc, saved_mc)
unexpected = {k: v for k, v in tc_diff.items() if k not in ALLOWED_DIFF}
print('training_config keys that differ (allowed):', sorted(k for k in tc_diff if k in ALLOWED_DIFF))
print('model_config diff:', mc_diff or 'none')
print('UNEXPECTED training_config diff:', json.dumps(unexpected, indent=1, default=str) if unexpected else 'none')
assert not mc_diff, 'model_config differs - the trainer would start a FRESH run. Do not launch.'
assert not unexpected, 'unexpected training_config differences - fix the flags before launching.'
COMMAND = [sys.executable, str(REPO / 'experiments' / 'train_universal.py')] + ARGV
print('\nPreflight OK. Expected first line:', EXPECTED_RESUME)
print('\n' + ' '.join(COMMAND))

In [ ]:
# @title 7) Resume training (kill-switch on fresh-run fallback)
import json
import time
assert 'COMMAND' in globals() and 'EXPECTED_RESUME' in globals(), 'run cell 6 (preflight) first'
ALLOC_ENV = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
FRESH_RUN_MARKERS = ('Could not resume', 'starting a fresh run')
resumed_ok, killed = False, False
with open(FAILURE_LOG, 'a') as logf:
    logf.write('==== Stage C attempt 4 (resume, rebuilt data) ' + time.strftime('%Y-%m-%d %H:%M:%S') + ' ====' + chr(10))
    logf.flush()
    proc = subprocess.Popen(COMMAND, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, env=ALLOC_ENV)
    for line in proc.stdout:
        print(line, end='')
        logf.write(line)
        logf.flush()
        if any(marker in line for marker in FRESH_RUN_MARKERS):
            proc.kill()
            killed = True
            logf.write('[notebook] killed trainer: resume failed, fresh-run fallback blocked' + chr(10))
            break
        if line.startswith('Resuming fine_tune'):
            resumed_ok = line.strip() == EXPECTED_RESUME
            if not resumed_ok:
                proc.kill()
                killed = True
                logf.write('[notebook] killed trainer: unexpected resume point' + chr(10))
                break
    proc.wait()
if killed:
    raise RuntimeError(
        'Trainer stopped by the notebook because it did not resume cleanly. Check that '
        f'{RUN4}/latest.pt is intact (compare with the cell-2 backup and restore it if needed) '
        'and rerun cells 6-7. Nothing was trained.')
status = json.loads((RUN4 / 'trainer_status.json').read_text())
print('returncode:', proc.returncode, '| trainer_status:', json.dumps(status, default=str)[:800])
if proc.returncode == 2 and status.get('status') == 'completed':
    print('Training finished all 50 epochs. Return code 2 = acceptance gate failed, which is '
          'EXPECTED here (mu ~0.2185 > 0.20 target); cell 8 performs the planned coherence decay.')
elif proc.returncode != 0:
    raise RuntimeError(f'Trainer exited with rc={proc.returncode}. If the log shows '
                       "'manifest rows failed to load', the Drive mount dropped: rerun cells 1, 6 and 7 "
                       '(training resumes from the last 25-step checkpoint). Log: ' + str(FAILURE_LOG))

In [ ]:
# @title 8) Post-run coherence decay on the epoch-50 checkpoint (mu < 0.20)
import json
import time
status = json.loads((RUN4 / 'trainer_status.json').read_text())
assert status.get('status') == 'completed', 'training has not finished all 50 epochs yet (run cell 7)'
FINAL_CHECKPOINT = RUN4 / 'latest.pt'
INCOHERENT = RUN4 / 'final_incoherent.pth'   # explicit output: never overwrite hashed inputs
ALLOC_ENV = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')

def _warm_status():
    if not INCOHERENT.is_file():
        return None
    return torch.load(INCOHERENT, map_location='cpu', weights_only=False).get('coherence_warmup', {}).get('status')

for attempt in range(3):
    if _warm_status() == 'target_met':
        break
    source = INCOHERENT if INCOHERENT.is_file() else FINAL_CHECKPOINT
    cmd = [sys.executable, str(REPO / 'scripts' / 'warmup_coherence.py'),
           '--init-checkpoint', str(source), '--output', str(INCOHERENT),
           '--steps', '2000', '--target', '0.20', '--learning-rate', '0.005',
           '--max-temperature', '0.005', '--check-every', '10', '--device', DEVICE]
    print('decay pass', attempt, ':', ' '.join(cmd))
    with open(FAILURE_LOG, 'a') as logf:
        logf.write('==== run4 post-run decay pass ' + str(attempt) + ' ' + time.strftime('%Y-%m-%d %H:%M:%S') + ' ====' + chr(10))
        proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=ALLOC_ENV)
        for line in proc.stdout:
            print(line, end='')
            logf.write(line)
        proc.wait()
    if proc.returncode not in (0, 2):
        raise RuntimeError(f'decay pass failed rc={proc.returncode}')
warm = torch.load(INCOHERENT, map_location='cpu', weights_only=False).get('coherence_warmup', {})
print('decay status:', warm.get('status'), '| final mu:', warm.get('final_mu'))
if warm.get('status') != 'target_met':
    raise RuntimeError('mu did not get below 0.20 after 3 passes - report to your senior before benchmarking.')

In [ ]:
# @title 9) Task 7.2c benchmark: decayed final checkpoint vs best.pt
for name, checkpoint in [('final_incoherent', RUN4 / 'final_incoherent.pth'), ('best', RUN4 / 'best.pt')]:
    output = RUN4 / f'benchmark_summary_{name}.json'
    cmd = [sys.executable, str(REPO / 'experiments' / 'eval_benchmarks.py'),
           '--checkpoint', str(checkpoint), '--data-root', str(DATA_ROOT),
           '--global-context-dim', '64', '--device', DEVICE, '--output', str(output)]
    print(' '.join(cmd))
    proc = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
    print('returncode:', proc.returncode)
    print(proc.stdout[-4000:])
    if proc.returncode != 0:
        print(proc.stderr[-4000:])
        raise RuntimeError(f'benchmark failed for {name}')
    print('wrote', output)